# CTA — GPU wall-clock validation (Test 1.1)

**Compositional Token Algebra** — reversible, parameter-free, session-scoped span
deduplication. This notebook answers the key infrastructure question:

> **Does the CTA speedup survive on a real GPU with FlashAttention?**
> (i.e. is the win eaten once attention is already IO-cheap?)

**Why it should survive (roofline):** CTA shortens the sequence `n -> m`, which
cuts *everything length-linear* (QKVO projections, MLP, LM head) — and FlashAttention
never touched that part. FA only makes the O(n²) attention *math* IO-cheap; it does
not reduce its FLOPs. On a 3B–7B model at 0.5k–8k, **70–98% of prefill compute is
length-linear**, so shrinking the token count wins in a plane orthogonal to FA.

**Protocol (Variant A — prefill-only):** we measure a single forward pass over the
prefix, `baseline (n tokens)` vs `CTA (m tokens)`, on GPU with the FlashAttention
backend (via SDPA or flash-attn), sweeping length. We report wall-clock, prefill
throughput (tok/s), the collapse ratio `m/n`, and — crucially — the **detection +
compose overhead** measured separately, so the number is honest.

**How to run:** `Runtime → Run all`. On free Colab (T4 16 GB) this defaults to
**Qwen2.5-3B**; on a ≥24 GB GPU it auto-selects **Qwen2.5-7B**. At the end it prints
a table and saves `results_gpu.json` — send that file back.


## 0 · Environment check (GPU, VRAM → auto-select model)

In [ ]:
import subprocess, torch, sys
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU! In Colab: Runtime → Change runtime type → GPU (T4)."
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM ~ {vram_gb:.1f} GB")

# Auto-select model by VRAM (bf16). 7B needs ~15 GB weights + activations.
if vram_gb >= 22:
    MODEL_NAME = "Qwen/Qwen2.5-7B"
elif vram_gb >= 14:
    MODEL_NAME = "Qwen/Qwen2.5-3B"
else:
    MODEL_NAME = "Qwen/Qwen2.5-1.5B"
print("Selected model:", MODEL_NAME)


## 1 · Install deps + fetch CTA source

We try to install `flash-attn`; if the CUDA build is slow/fails we silently fall
back to PyTorch **SDPA**, whose GPU backend *is* FlashAttention. Either way the
attention path is a fused FA kernel — which is exactly the condition we're testing.

In [ ]:
import os, sys, subprocess

def sh(cmd):
    print("$", cmd); return subprocess.run(cmd, shell=True).returncode

# transformers + accelerate (Qwen2 needs >=4.40)
sh("pip -q install -U 'transformers>=4.44' accelerate >/dev/null 2>&1")

# Try flash-attn (optional). Comment out to skip; SDPA fallback is fine.
USE_FLASH_ATTN = False   # set True to attempt the (slow) flash-attn build
if USE_FLASH_ATTN:
    rc = sh("pip -q install flash-attn --no-build-isolation")
    HAS_FA = (rc == 0)
else:
    HAS_FA = False
print("flash-attn installed:", HAS_FA, "(otherwise SDPA/FlashAttention backend is used)")

# Fetch CTA source (algebra.py, detector.py) from the public repo.
if not os.path.isdir("compositional-token-algebra"):
    sh("git clone --depth 1 https://github.com/VadShv/compositional-token-algebra")
sys.path.insert(0, "compositional-token-algebra/src")
from cta.algebra import compose, decompose               # noqa: E402
from cta.detector import select_collapse_spans, build_segments  # noqa: E402
print("CTA operators imported OK")


## 2 · Load model with a FlashAttention backend

`attn_implementation="sdpa"` gives the fused FlashAttention kernel on GPU (or
`"flash_attention_2"` if flash-attn built). bf16, eval, no grad.

In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

attn_impl = "flash_attention_2" if HAS_FA else "sdpa"
print("attn_implementation =", attn_impl)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16,
    attn_implementation=attn_impl, device_map="cuda")
model.eval()
cfg = model.config
DEV = "cuda"
print(f"loaded: layers={cfg.num_hidden_layers} d={cfg.hidden_size} "
      f"heads={cfg.num_attention_heads} kv={getattr(cfg,'num_key_value_heads','?')} "
      f"vocab={cfg.vocab_size} ctx={cfg.max_position_embeddings}")


## 3 · Build a repetition-heavy prefix + forward helpers

CTA's win is content-dependent — it needs *deep* redundancy to collapse. We build
the corpus from the repo's own source files (code = deep redundancy), matching the
CPU experiments. The forward path is architecture-agnostic: we run on **input
embeddings**, so `model.model(inputs_embeds=...) + lm_head` — identical to the CPU
port, no algorithm change.

In [ ]:
import glob, torch

# repetition-heavy text: concatenate repo python files (code has deep redundancy)
paths = sorted(glob.glob("compositional-token-algebra/src/cta/*.py") +
               glob.glob("compositional-token-algebra/experiments/*.py"))
text = "\n\n".join(open(p).read() for p in paths)
# amplify redundancy a bit so collapse candidates exist at every length
text = (text + "\n\n") * 6
all_ids = tok(text, return_tensors="pt").input_ids.squeeze(0).to(DEV)
print("corpus tokens available:", all_ids.numel())

@torch.no_grad()
def emb_of(ids):
    return model.get_input_embeddings()(ids.unsqueeze(0)).squeeze(0)  # [n,d]

@torch.no_grad()
def baseline_forward(ids):
    emb = emb_of(ids)
    h = model.model(inputs_embeds=emb.unsqueeze(0)).last_hidden_state
    _ = model.lm_head(h[:, -1:, :])          # only need last-token logits for prefill timing
    return ids.shape[0]

@torch.no_grad()
def build_collapsed_emb(ids, segments, mode="norm"):
    '''Vectorized-as-possible collapse on GPU embeddings. Returns [m,d].'''
    emb = emb_of(ids)
    vecs = []
    for (s, e, is_col) in segments:
        if is_col:
            e_bar, R, pi = compose(emb[s:e], mode=mode)
            vecs.append(e_bar)
        else:
            vecs.append(emb[s])
    return torch.stack(vecs, 0)              # [m,d]

@torch.no_grad()
def cta_forward(collapsed):
    h = model.model(inputs_embeds=collapsed.unsqueeze(0)).last_hidden_state
    _ = model.lm_head(h[:, -1:, :])
    return collapsed.shape[0]


## 4 · Reversibility on real large-model embeddings

Same L∞-error check as the CPU/0.5B runs, now on this model's embedding space.
Must hold at machine epsilon (~1e-8 in bf16-cast-to-fp32; we compose in fp32).

In [ ]:
import torch
emb_full = emb_of(all_ids[:2048]).float()   # fp32 for a clean reversibility number
g = torch.Generator(device=DEV).manual_seed(0)
rev = {}
for mode in ["uniform","norm","posdecay","selfconsist"]:
    max_err = 0.0
    for _ in range(200):
        k = int(torch.randint(2,16,(1,),generator=g,device=DEV).item())
        start = int(torch.randint(0, emb_full.shape[0]-k,(1,),generator=g,device=DEV).item())
        span = emb_full[start:start+k]
        e_bar,R,pi = compose(span, mode=mode)
        recon = decompose(e_bar,R)
        max_err = max(max_err,(recon-span).abs().max().item())
    rev[mode] = max_err
    print(f"  {mode:12s} max L-inf err = {max_err:.2e}")


## 5 · Wall-clock: baseline vs CTA, sweep length (the actual Test 1.1)

Proper GPU timing: warmup + `torch.cuda.synchronize()` + median of repeats.
We separately time **detection+compose overhead** (CPU-side span selection + the
Python collapse loop) so the reported speedup is honest — an infra reviewer will
ask "did you hide the preprocessing cost?" We report both:
  * `speedup_fwd`   — forward-only (baseline fwd / CTA fwd), the compute ceiling
  * `speedup_total` — including detection+compose overhead (the real end-to-end)

In [ ]:
import time, torch, statistics

def cuda_time(fn, repeats=5, warmup=2):
    for _ in range(warmup):
        fn(); torch.cuda.synchronize()
    ts=[]
    for _ in range(repeats):
        torch.cuda.synchronize(); t0=time.perf_counter()
        fn(); torch.cuda.synchronize()
        ts.append(time.perf_counter()-t0)
    return statistics.median(ts)

# lengths capped by available corpus + model context
LENGTHS = [L for L in [512,1024,2048,4096,8192] if L <= all_ids.numel() and L <= cfg.max_position_embeddings]
rows=[]
for L in LENGTHS:
    ids = all_ids[:L]
    # --- detection + segment build overhead (CPU) ---
    id_list = ids.tolist()
    t_det0 = time.perf_counter()
    spans = select_collapse_spans(id_list, k_min=3, k_max=16, f_min=2)
    segs  = build_segments(id_list, spans)
    t_det = time.perf_counter() - t_det0
    m = len(segs); n_col = sum(1 for *_ , c in segs if c)
    if m >= L:
        print(f"  L={L}: no collapse candidates, skip"); continue

    # --- compose overhead (GPU collapse loop) ---
    t_comp = cuda_time(lambda: build_collapsed_emb(ids, segs), repeats=3, warmup=1)
    collapsed = build_collapsed_emb(ids, segs)

    # --- forward timings ---
    t_base = cuda_time(lambda: baseline_forward(ids))
    t_cta  = cuda_time(lambda: cta_forward(collapsed))

    sp_fwd   = t_base / t_cta
    sp_total = t_base / (t_cta + t_comp + t_det)
    tps_base = L / t_base
    tps_cta  = m / t_cta
    row = dict(len=L, m=m, collapsed_spans=n_col, ratio=round(m/L,3),
               baseline_ms=round(t_base*1000,1), cta_ms=round(t_cta*1000,1),
               compose_ms=round(t_comp*1000,1), detect_ms=round(t_det*1000,1),
               speedup_fwd=round(sp_fwd,3), speedup_total=round(sp_total,3),
               tok_s_base=round(tps_base,0), tok_s_cta=round(tps_cta,0))
    rows.append(row)
    print(f"  L={L:5d} m={m:5d} (r={m/L:.2f})  base={row['baseline_ms']:.1f}ms "
          f"cta={row['cta_ms']:.1f}ms  +det={row['detect_ms']:.1f} +comp={row['compose_ms']:.1f}  "
          f"| fwd {sp_fwd:.2f}x  total {sp_total:.2f}x")


## 6 · Save results → send `results_gpu.json` back

In [ ]:
import json, torch
out = dict(
    model=MODEL_NAME, attn_impl=attn_impl, flash_attn=HAS_FA,
    gpu=torch.cuda.get_device_name(0),
    n_layer=cfg.num_hidden_layers, d=cfg.hidden_size, vocab=cfg.vocab_size,
    reversibility=rev, rows=rows,
)
with open("results_gpu.json","w") as f: json.dump(out, f, indent=2)
print(json.dumps(out, indent=2))
print("\nSaved results_gpu.json — download it (Files panel, left) and send it back.")
try:
    from google.colab import files; files.download("results_gpu.json")
except Exception as e:
    print("(auto-download unavailable:", e, "— grab it from the Files panel)")


## How to read the result (go / no-go for Test 1.1)

**GO** if `speedup_fwd` rises with length and the *total* speedup crosses 1.0 and
grows (e.g. ~1.0× at 512 → >1.3× at 4k–8k). That confirms the CTA win survives the
FlashAttention backend, because it rides on shrinking the length-linear compute
(MLP/projections/head), which FA never optimized.

**Expected shape** (from the roofline): near-flat/modest at 512 (overhead ≈ gain,
GPU underutilized), then rising with length. If detection/compose overhead
dominates at short lengths, that is a known, fixable engineering cost (vectorize the
collapse loop / move detection off the hot path) — not a refutation of the mechanism.

**NO-GO / nuance** if `speedup_fwd` itself stays ≤1.0 even at 4k–8k — that would mean
the GPU is so underutilized that fewer tokens don't help, and we'd re-test at larger
batch or longer context before concluding.
